# Calculate the checkout test's lift

*Data Analysis in Python · Topic 9.1*

The checkout redesign test has ended, and the product manager wants the headline numbers for all six weeks: each variant's conversion rate and revenue per user, and B's lift on both.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the test's two files: `assignments`, one row per user in the test, and `conversions`, one row per user who converted.
2. In the first cell marked ✏️, create the `users` table: the `assignments` table with each user's `revenue` from the `conversions` table, a `converted` column that's `True` for users with a conversion, and `revenue` 0 for users without one.
3. In the second cell marked ✏️, create the `results` table, one row per variant, with the columns:
   - `users` and `conversions`: the number of users and of converted users
   - `conversion_rate`: conversions divided by users, as a fraction
   - `revenue_per_user`: the variant's revenue divided by its users
4. In the last cell marked ✏️, create `conversion_lift` and `revenue_lift`: B's lift over A on each metric, as fractions (0.05 for +5%).
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `users` table has 80,054 rows, 3,851 of them converted. The `results` table has A with 41,706 users, 1,907 conversions, a conversion rate of 0.0457 and $4.06 revenue per user, and B with 38,348 users, 1,944 conversions, 0.0507 and $4.45. `conversion_lift` is about `0.1087`, and `revenue_lift` about `0.0986`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"

In [ ]:
# The test's two files, with the user IDs as text. Run it, and leave it as it is.
assignments = pd.read_csv(DATA_URL + "ab_test_assignments.csv", dtype={"user_pseudo_id": str}, parse_dates=["assigned_at"])
conversions = pd.read_csv(DATA_URL + "ab_test_conversions.csv", dtype={"user_pseudo_id": str}, parse_dates=["converted_at"])
assignments.shape, conversions.shape

## One row per user

In [ ]:
# ✏️ Create the users table


## The variants

In [ ]:
# ✏️ Create the results table


## The lift

In [ ]:
# ✏️ Create conversion_lift and revenue_lift


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    users = found.get("users")
    if not isinstance(users, pd.DataFrame) or not {"variant", "converted", "revenue"} <= set(users.columns):
        results.append((False, "users should be a DataFrame with the assignments' columns, revenue and converted: assignments.merge(conversions[[\"user_pseudo_id\", \"revenue\"]], on=\"user_pseudo_id\", how=\"left\", validate=\"one_to_one\")."))
    elif len(users) == 3851:
        results.append((False, "The users table has 3,851 rows: only the users who converted. An inner join drops everyone else, so every rate would be 100%. Join with how=\"left\" to keep every assigned user."))
    elif len(users) != 80054:
        results.append((False, f"The users table should have one row per assigned user, 80,054. It has {len(users):,}."))
    elif users["revenue"].isna().any():
        results.append((False, f"The users table's revenue has {users['revenue'].isna().sum():,} missing values: the users who didn't convert. For revenue per user they spent $0: users[\"revenue\"].fillna(0)."))
    elif users["converted"].sum() != 3851:
        results.append((False, f"converted should be True for the 3,851 users with a conversion. It's True for {users['converted'].sum():,}. Set it before you fill revenue: users[\"revenue\"].notna()."))
    else:
        results.append((True, "One row for each of the 80,054 users in the test, 3,851 of them converted, and $0 for the rest."))

    table = found.get("results")
    columns = {"users", "conversions", "conversion_rate", "revenue_per_user"}
    if not isinstance(table, pd.DataFrame) or not columns <= set(table.columns) or not {"A", "B"} <= set(table.index):
        results.append((False, "results should be a DataFrame with the variants A and B as its index, and the columns users, conversions, conversion_rate and revenue_per_user."))
    else:
        rate_a, rate_b = table.loc["A", "conversion_rate"], table.loc["B", "conversion_rate"]
        rpu_a, rpu_b = table.loc["A", "revenue_per_user"], table.loc["B", "revenue_per_user"]
        if table.loc["A", "users"] != 41706 or table.loc["B", "users"] != 38348:
            results.append((False, f"results' users should be 41,706 for A and 38,348 for B. They are {table.loc['A', 'users']:,} and {table.loc['B', 'users']:,}."))
        elif abs(rate_a - 1) < 0.0001:
            results.append((False, "Every conversion rate is 1: the users table has only the converters. Join with how=\"left\"."))
        elif abs(rate_a - 4.5725) < 0.001:
            results.append((False, "results' conversion_rate is in percent: A's is 4.57. Keep rates as fractions, 0.0457."))
        elif abs(rate_a - 0.045725) > 0.000001 or abs(rate_b - 0.050694) > 0.000001:
            results.append((False, f"conversion_rate should be conversions divided by users: 0.0457 for A and 0.0507 for B. It is {rate_a:.4f} and {rate_b:.4f}."))
        elif abs(rpu_a - 88.69) < 0.01:
            results.append((False, "revenue_per_user is $88.69 for A: that's the average order value, the revenue divided by the buyers only. Divide by all the variant's users, with $0 for those who didn't buy."))
        elif abs(rpu_a - 4.0552) > 0.0001 or abs(rpu_b - 4.4549) > 0.0001:
            results.append((False, f"revenue_per_user should be the variant's revenue divided by its users: $4.06 for A and $4.45 for B. It is {rpu_a:.2f} and {rpu_b:.2f}."))
        else:
            results.append((True, f"A converted {rate_a:.2%} of its users and B {rate_b:.2%}, with ${rpu_a:.2f} and ${rpu_b:.2f} of revenue per user."))

    conversion_lift, revenue_lift = found.get("conversion_lift"), found.get("revenue_lift")
    if not number(conversion_lift) or not number(revenue_lift):
        results.append((False, "conversion_lift and revenue_lift should be numbers: results.loc[\"B\", \"conversion_rate\"] / results.loc[\"A\", \"conversion_rate\"] - 1, and the same for revenue_per_user."))
    elif abs(conversion_lift + 0.0980) < 0.0001:
        results.append((False, "conversion_lift is -0.0980: that's A relative to B. Lift is the new version relative to the control: B / A - 1."))
    elif abs(conversion_lift - 0.004969) < 0.000001 or abs(conversion_lift - 0.4969) < 0.0001:
        results.append((False, f"conversion_lift is {conversion_lift:.4g}: that's the difference in rates, in {'percentage points' if conversion_lift > 0.1 else 'fractions'}. Lift is relative: B's rate divided by A's, minus 1."))
    elif abs(conversion_lift - 10.8668) < 0.001:
        results.append((False, "conversion_lift is in percent, 10.87. Keep it as a fraction, 0.1087, and format it when you show it."))
    elif abs(conversion_lift - 0.108668) > 0.000001:
        results.append((False, f"conversion_lift should be B's conversion rate divided by A's, minus 1: about 0.1087. It is {conversion_lift:.4f}."))
    elif abs(revenue_lift + 0.009119) < 0.0001:
        results.append((False, "revenue_lift is -0.0091: that's the lift in average order value, the buyers only. Use revenue_per_user, with every user in the denominator."))
    elif abs(revenue_lift - 0.098558) > 0.000001:
        results.append((False, f"revenue_lift should be B's revenue per user divided by A's, minus 1: about 0.0986. It is {revenue_lift:.4f}."))
    else:
        results.append((True, f"B's conversion rate is {conversion_lift:+.1%} on A's, +0.50 percentage points, and its revenue per user {revenue_lift:+.1%}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
users = assignments.merge(conversions[["user_pseudo_id", "revenue"]], on="user_pseudo_id", how="left", validate="one_to_one")
users["converted"] = users["revenue"].notna()
users["revenue"] = users["revenue"].fillna(0)
users.shape, users["converted"].sum()
```

```python
results = users.groupby("variant").agg(users=("user_pseudo_id", "count"), conversions=("converted", "sum"), revenue=("revenue", "sum"))
results["conversion_rate"] = results["conversions"] / results["users"]
results["revenue_per_user"] = results["revenue"] / results["users"]
results
```

```python
conversion_lift = results.loc["B", "conversion_rate"] / results.loc["A", "conversion_rate"] - 1
revenue_lift = results.loc["B", "revenue_per_user"] / results.loc["A", "revenue_per_user"] - 1
conversion_lift, revenue_lift
```

</details>